# Section 4 · Notebook 2 — Agent as Tool: Tutoring Assistant

A teacher agent receives a student's question and delegates to the right specialist tutor (math, writing, science). The teacher always stays in control.

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: /private/tmp/scaler-code-venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using amazon.nova-lite-v1:0 in us-east-1


In [3]:
from strands import Agent

## Specialist agents

Each specialist gets a `name` and a `description`. When an agent is used as a tool, the name becomes the tool name and the description tells the teacher when to call it.

In [4]:
math_tutor = Agent(
    name="math_tutor",
    description="Solves mathematical equations and explains math concepts step by step",
    model=model,
    system_prompt="You are a math tutor. Solve math problems and explain your solutions clearly, step by step.",
    callback_handler=None,
)

writing_coach = Agent(
    name="writing_coach",
    description="Helps with writing, grammar, and essay structure",
    model=model,
    system_prompt="You are a writing coach. Help with writing, grammar, and essay structure. Provide constructive feedback.",
    callback_handler=None,
)

science_helper = Agent(
    name="science_helper",
    description="Explains scientific concepts and principles in simple terms",
    model=model,
    system_prompt="You are a science tutor. Explain scientific concepts clearly using simple language and real-world examples.",
    callback_handler=None,
)

## The teacher (orchestrator)

Pass the specialist agents straight into `tools=[...]` — Strands wraps each one as a callable tool.

In [5]:
teacher = Agent(
    model=model,
    system_prompt="""You are a teacher who helps students by delegating to specialist tutors.
Never solve problems yourself. Always delegate to the appropriate specialist:
- math_tutor: for math problems and equations
- writing_coach: for writing, grammar, and essays
- science_helper: for science questions

After receiving the specialist's response, present it to the student in a friendly way.""",
    tools=[math_tutor, writing_coach, science_helper],
)

## Test with different questions

In [6]:
# Math → math_tutor
response = teacher("Can you help me solve this quadratic equation: x^2 + 5x + 6 = 0?")

<thinking>The User has asked for help with a quadratic equation. This falls under the math_tutor's expertise. I will delegate this to the math_tutor.</thinking>

Tool #1: math_tutor
Hi there! I've asked the math_tutor to help us solve the quadratic equation \( x^2 + 5x + 6 = 0 \). Here's the step-by-step solution provided by the math_tutor:

1. **Identify the quadratic equation**: The given equation is \( x^2 + 5x + 6 = 0 \).
2. **Factor the quadratic expression**: We need to find two numbers that multiply to the constant term (6) and add up to the coefficient of the linear term (5).
   
   The pairs of numbers that multiply to 6 are:
   - \( 1 \times 6 = 6 \)
   - \( 2 \times 3 = 6 \)
   - \( -1 \times -6 = 6 \)
   - \( -2 \times -3 = 6 \)

   The pair that adds up to 5 is \( 2 \) and \( 3 \).

3. **Rewrite the quadratic expression using these numbers**:
   \[
   x^2 + 5x + 6 = (x + 2)(x + 3)
   \]

4. **Set each factor equal to zero**:
   \[
   (x + 2)(x + 3) = 0
   \]

5. **Solve fo

In [7]:
# Writing → writing_coach
response = teacher("How do I write a strong thesis statement for my essay about climate change?")

Writing a strong thesis statement for an essay about climate change involves several key steps:

1. **Identify Your Focus**: Determine the specific aspect of climate change you want to explore, such as its impact on ecosystems, economic consequences, or policy responses.

2. **Take a Clear Position**: Your thesis should state a clear, debatable position. For example, you might argue that renewable energy solutions are crucial for mitigating climate change.

3. **Be Specific**: Provide enough detail to give readers a clear idea of your argument. Avoid vague statements.

4. **Reflect the Scope of Your Essay**: Ensure your thesis reflects the breadth and depth of your essay. If you’re discussing multiple causes or effects, make sure your thesis encompasses this.

**Example Thesis Statements**:

- "Implementing aggressive renewable energy policies is essential for mitigating the adverse effects of climate change on global ecosystems."
  
- "Climate change poses significant economic risks, 

In [8]:
# Science → science_helper
response = teacher("Why does ice float on water?")

Ice floats on water due to its lower density compared to liquid water. This unique property arises from the molecular structure of water and the hydrogen bonds that form between water molecules.

In the liquid state, water molecules are closely packed together, allowing them to occupy a smaller volume. However, when water freezes and turns into ice, the hydrogen bonds between water molecules create a hexagonal lattice structure. This arrangement causes the molecules to be more spread out, resulting in a larger volume for a given mass of ice.

As a result, ice has a density of approximately 0.92 grams per cubic centimeter, while liquid water has a density of about 1 gram per cubic centimeter. Since ice is less dense than liquid water, it floats when placed in water. This phenomenon is crucial for aquatic life, as it allows ice to form on the surface of bodies of water during winter, creating an insulating layer that protects the water below from freezing solid.